# Problem Set 2. Neural networks & optimisation

**Course:** Analysis of Unstructured Data  
**Instructor:** Janusz Szwabiński  
**Goal:** Derive the deep-learning backbone once, so it can be reused for text, images and audio.  

---


In [1]:
# Setup environment & imports
import sys
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
%config InlineBackend.figure_format = 'retina'


### Problem 1 [Theory] (Softmax + cross-entropy)


**Task:** For logits $z \in \mathbb{R}^K$, softmax $p_k = \frac{e^{z_k}}{\sum_j e^{z_j}}$ and one-hot target $y$, show that the cross-entropy loss $L = -\sum_k y_k \log p_k$ has gradient $\frac{\partial L}{\partial z} = p-y$. Explain why this clean form is the reason softmax and cross-entropy are always paired.

---

### Derivation

**Step 1: Define softmax and cross-entropy**

The softmax function converts logits into class probabilities:

$$
p_k = \frac{e^{z_k}}{\sum_{j=1}^{K}e^{z_j}}
$$

The cross-entropy loss for a one-hot target vector $y$, where $\sum_{k=1}^{K}y_k=1$, is:

$$
L=-\sum_{k=1}^{K}y_k\log p_k
$$

**Step 2: Substitute softmax into the loss function**

$$
L=-\sum_{k=1}^{K}y_k
\log\left(\frac{e^{z_k}}{\sum_{j=1}^{K}e^{z_j}}\right)
$$

Using logarithm properties and $\sum_{k=1}^{K}y_k=1$:

$$
L=-\sum_{k=1}^{K}y_kz_k+
\log\left(\sum_{j=1}^{K}e^{z_j}\right)
$$

**Step 3: Compute the gradient with respect to $z_i$**

Differentiating both terms:

$$
\frac{\partial L}{\partial z_i}
=
-\frac{\partial}{\partial z_i}\sum_{k=1}^{K}y_kz_k
+
\frac{\partial}{\partial z_i}
\log\left(\sum_{j=1}^{K}e^{z_j}\right)
$$

The first term gives $-y_i$. Applying the chain rule to the second term:

$$
\frac{\partial L}{\partial z_i}
=
-y_i+\frac{e^{z_i}}{\sum_{j=1}^{K}e^{z_j}}
$$

Since the fraction is the softmax probability $p_i$:

$$
\frac{\partial L}{\partial z_i}=p_i-y_i
$$

Therefore, in vector form:

$$
\nabla_z L=p-y
$$

---

### Why Are Softmax and Cross-Entropy Paired?

Softmax converts logits into probabilities, while cross-entropy measures how well these probabilities match the true labels.

Their combination produces the simple gradient $\nabla_z L=p-y$, which:

- Simplifies backpropagation by avoiding explicit computation of the softmax Jacobian.
- Provides a direct error signal: predicted probabilities minus true labels.
- Enables efficient gradient-based optimization.

**Conclusion:** Softmax and cross-entropy are commonly paired because their combination provides a simple and computationally efficient gradient for training multi-class classification models.


### Problem 2 [Theory] (Backpropagation)

**Task:** Consider a two-layer MLP $h = \sigma(W_1x + b_1)$, $\hat{y} = \operatorname{softmax}(W_2h + b_2)$ with cross-entropy loss. Derive $\frac{\partial L}{\partial W_2}$, $\frac{\partial L}{\partial b_2}$, $\frac{\partial L}{\partial W_1}$, $\frac{\partial L}{\partial b_1}$ explicitly using the chain rule, for a general elementwise activation $\sigma$.

---

### Derivation

**Step 1: Define the forward pass**

Let $x\in\mathbb{R}^d$ be the input, $h\in\mathbb{R}^m$ the hidden-layer output, and $\hat{y}\in\mathbb{R}^K$ the predicted class probabilities.

The forward pass is defined as:

$$
a_1=W_1x+b_1,\qquad h=\sigma(a_1)
$$

$$
a_2=W_2h+b_2,\qquad \hat{y}=\operatorname{softmax}(a_2)
$$

where $\sigma$ is an elementwise activation function and:

$$
W_1\in\mathbb{R}^{m\times d},\quad b_1\in\mathbb{R}^m,
\quad W_2\in\mathbb{R}^{K\times m},\quad b_2\in\mathbb{R}^K
$$

The cross-entropy loss for a one-hot target vector $y$ is:

$$
L=-\sum_{k=1}^{K}y_k\log\hat{y}_k
$$

**Step 2: Compute the gradients for the output layer**

From Problem 1, the gradient of softmax combined with cross-entropy is:

$$
\delta_2=\frac{\partial L}{\partial a_2}=\hat{y}-y
$$

Since $a_2=W_2h+b_2$, we apply the chain rule elementwise. For each weight $(W_2)_{kj}$:

$$
\frac{\partial L}{\partial (W_2)_{kj}}
=
\frac{\partial L}{\partial (a_2)_k}
\frac{\partial (a_2)_k}{\partial (W_2)_{kj}}
=
(\delta_2)_k h_j
$$

Therefore:

$$
\frac{\partial L}{\partial W_2}=\delta_2h^\top=(\hat{y}-y)h^\top
$$

Similarly, since $\frac{\partial (a_2)_k}{\partial (b_2)_k}=1$:

$$
\frac{\partial L}{\partial b_2}=\delta_2=\hat{y}-y
$$

**Step 3: Backpropagate the error to the hidden layer**

Using the chain rule, the gradient with respect to the hidden-layer output is:

$$
\frac{\partial L}{\partial h}=W_2^\top\delta_2
$$

Since $h=\sigma(a_1)$ and $\sigma$ acts elementwise:

$$
\frac{\partial h}{\partial a_1}
=\operatorname{diag}(\sigma'(a_1))
$$

Thus, the hidden-layer error is:

$$
\delta_1=\frac{\partial L}{\partial a_1}
=
\left(W_2^\top\delta_2\right)\odot\sigma'(a_1)
$$

where $\odot$ denotes elementwise multiplication.

**Step 4: Compute the gradients for the first layer**

Since $a_1=W_1x+b_1$, applying the chain rule to each weight $(W_1)_{ij}$:

$$
\frac{\partial L}{\partial (W_1)_{ij}}
=
\frac{\partial L}{\partial (a_1)_i}
\frac{\partial (a_1)_i}{\partial (W_1)_{ij}}
=
(\delta_1)_i x_j
$$

Therefore:

$$
\frac{\partial L}{\partial W_1}=\delta_1x^\top
$$

Similarly, for the bias vector:

$$
\frac{\partial L}{\partial b_1}=\delta_1
$$

---

### Final Results

Define the output-layer and hidden-layer error vectors:

$$
\delta_2=\hat{y}-y
$$

$$
\delta_1=\left(W_2^\top\delta_2\right)\odot\sigma'(W_1x+b_1)
$$

The four required gradients are:

$$
\begin{aligned}
\frac{\partial L}{\partial W_2}&=\delta_2h^\top\\[6pt]
\frac{\partial L}{\partial b_2}&=\delta_2\\[6pt]
\frac{\partial L}{\partial W_1}&=\delta_1x^\top\\[6pt]
\frac{\partial L}{\partial b_1}&=\delta_1
\end{aligned}
$$

These equations describe backpropagation: the prediction error is first computed at the output layer and then propagated backward through the network using the chain rule.


### Problem 3 [Theory] 

**Task:** Write the update rules of SGD, SGD+momentum and Adam. Show that momentum is an exponentially moving average of gradients, and explain the purpose of Adam's bias-correction terms $\hat{m}_t$, $\hat{v}_t$ in the first few steps.

---

### Assumptions & Notation

Let $\theta_t$ denote the model parameters at iteration $t$, and let:

$$
g_t=\nabla_\theta L(\theta_{t-1})
$$

be the gradient of the loss function computed on the current mini-batch.

We use:
- $\eta>0$ — learning rate,
- $\mu\in[0,1)$ — momentum coefficient,
- $\beta_1,\beta_2\in[0,1)$ — exponential decay coefficients in Adam,
- $\varepsilon>0$ — small constant for numerical stability.

---

### 1. Stochastic Gradient Descent (SGD)

SGD updates the parameters in the direction opposite to the gradient:

$$
\theta_t=\theta_{t-1}-\eta g_t
$$

Each update depends only on the gradient calculated at the current iteration.

---

### 2. SGD with Momentum

Momentum introduces a velocity vector that accumulates information from previous gradients:

$$
v_0=0
$$

$$
v_t=\mu v_{t-1}+(1-\mu)g_t
$$

$$
\theta_t=\theta_{t-1}-\eta v_t
$$

**Showing that momentum is an exponential moving average:**

Expanding the recursive formula:

$$
v_t=\mu v_{t-1}+(1-\mu)g_t
$$

Substituting $v_{t-1}$:

$$
v_t=\mu[\mu v_{t-2}+(1-\mu)g_{t-1}]+(1-\mu)g_t
$$

$$
v_t=\mu^2v_{t-2}+\mu(1-\mu)g_{t-1}+(1-\mu)g_t
$$

Continuing recursively and using $v_0=0$:

$$
v_t=(1-\mu)\sum_{i=1}^{t}\mu^{t-i}g_i
$$

The weights $(1-\mu)\mu^{t-i}$ decrease exponentially for older gradients.

Therefore, momentum is an **exponentially weighted moving average (EMA)** of past gradients (initially biased toward zero because $v_0=0$).

This smooths noisy gradient updates and helps accelerate optimization in directions where gradients are consistent.

---

### 3. Adam (Adaptive Moment Estimation)

Adam combines momentum with adaptive learning rates based on the first and second moments of the gradients.

**First moment (mean of gradients):**

$$
m_0=0,\qquad
m_t=\beta_1m_{t-1}+(1-\beta_1)g_t
$$

**Second moment (mean of squared gradients):**

$$
s_0=0,\qquad
s_t=\beta_2s_{t-1}+(1-\beta_2)g_t^2
$$

where $g_t^2$ denotes elementwise squaring.

**Bias correction:**

$$
\hat{m}_t=\frac{m_t}{1-\beta_1^t}
$$

$$
\hat{v}_t=\frac{s_t}{1-\beta_2^t}
$$

**Parameter update:**

$$
\theta_t=\theta_{t-1}
-\eta\frac{\hat{m}_t}{\sqrt{\hat{v}_t}+\varepsilon}
$$

All divisions and square roots are performed elementwise.

---

### 4. Why Does Adam Need Bias Correction?

Both moving averages are initialized at zero:

$$
m_0=0,\qquad s_0=0
$$

Consequently, their estimates are biased toward zero during the first iterations.

For example, at $t=1$:

$$
m_1=(1-\beta_1)g_1
$$

$$
s_1=(1-\beta_2)g_1^2
$$

With the typical values $\beta_1=0.9$ and $\beta_2=0.999$:

$$
m_1=0.1g_1,\qquad s_1=0.001g_1^2
$$

Thus, both estimates are much smaller than the corresponding gradient quantities.

Applying the bias corrections:

$$
\hat{m}_1=\frac{m_1}{1-\beta_1}
=\frac{(1-\beta_1)g_1}{1-\beta_1}
=g_1
$$

$$
\hat{v}_1=\frac{s_1}{1-\beta_2}
=\frac{(1-\beta_2)g_1^2}{1-\beta_2}
=g_1^2
$$

More generally, assuming a constant expected gradient $\mathbb{E}[g_t]=G$:

$$
\mathbb{E}[m_t]=(1-\beta_1^t)G
$$

Therefore:

$$
\mathbb{E}[\hat{m}_t]
=\frac{\mathbb{E}[m_t]}{1-\beta_1^t}=G
$$

The same reasoning applies to the second moment estimate.

The factors $1-\beta_1^t$ and $1-\beta_2^t$ correct the initialization bias. As $t$ increases:

$$
\beta_1^t\to0,\qquad\beta_2^t\to0
$$

so the correction factors approach $1$ and become less important.

---

### Conclusion

- **SGD** updates parameters using only the current gradient.
- **SGD with momentum** uses an exponential moving average of past gradients to smooth updates.
- **Adam** combines first- and second-moment estimates to adapt the update magnitude for each parameter.
- **Bias correction** compensates for the zero initialization of Adam's moving averages, especially during the first iterations.

### Problem 4 [Python]

**Task:**
Implement the two-layer MLP of Problem 2 in pure NumPy, including the backward pass. Verify every gradient by central finite differences (gradient check to < 10^-5 relative error).


In [ ]:
# Problem 4 [Python]
import numpy as np

# TODO: Implement 2-layer MLP in pure NumPy with backpropagation and gradient check


### Problem 5 [Python]

**Task:**
Re-implement the same network in PyTorch with autograd. Train it on the Problem-Set-1 feature set and reproduce the NumPy loss curve to confirm the implementations agree.


In [ ]:
# Problem 5 [Python]
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

# TODO: PyTorch autograd implementation matching NumPy MLP


### Problem 6 [Python]

**Task:**
Study regularisation: train with/without L2, dropout and early stopping; produce train/validation learning curves for each and a small table of final accuracies.


In [ ]:
# Problem 6 [Python]
# TODO: Study regularisation (L2, dropout, early stopping) and plot curves


### Problem 7 [Analysis]

---
**Discussion:**
